# RAG 팀 최종 공통 생성 + 평가 Notebook

이 Notebook은 **LLM-only 팀과 LLM+RAG 팀을 같은 기준으로 비교**하기 위한 RAG 팀용 최종본입니다.

## 비교 원칙

### 두 팀이 반드시 같아야 하는 것
- 동일한 `validation_canonical_981.csv`
- 동일한 981개 질문과 `reference_responses_json`
- `max_new_tokens = 300`
- `do_sample = False`
- 동일한 공통 평가 코드
- 동일한 Semantic 평가 모델
- 동일한 Key Fact 추출 규칙
- 동일한 Multi-reference 방식

### RAG 팀만 달라지는 것
- Retriever / Embedding / Vector DB
- 검색된 정책 Context
- RAG 전용 System Prompt의 정책문서 지시

즉, **평가 기준은 같고 답변 생성 파이프라인만 다르게** 유지합니다.

> 주의: 정답 응답이나 정답 정책 chunk를 직접 넣으면 실제 RAG 평가가 아닙니다.

In [ ]:
# ============================================================
# 1. 라이브러리
# ============================================================

import gc
import json
import time
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 2. 팀원이 수정할 설정 (경로 및 모델 설정 완료)
# ============================================================

# 실험 이름
MODEL_NAME = "RAG_BGE_M3_RERANKER_LORA"

# 프로젝트 루트 폴더 자동 인식 (로컬 / 코랩 겸용)
PROJECT_DIR = Path(".").resolve()

# ----------------------------------------------------------------
# LLM
# ----------------------------------------------------------------
MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
MODEL_REVISION = "0ff6b5ec7c13b049b253a16a889aa269e6b79a94"

# 튜닝팀에게 전달받은 LoRA Adapter 폴더 경로
ADAPTER_DIR = PROJECT_DIR / "models" / "lora_adapter"

# ----------------------------------------------------------------
# 공통 평가 데이터 (data/csv 폴더 우선 탐색)
# ----------------------------------------------------------------
CANONICAL_CSV = PROJECT_DIR / "data" / "csv" / "validation_canonical_981.csv"
if not CANONICAL_CSV.exists():
    CANONICAL_CSV = PROJECT_DIR / "data" / "validation_canonical_981.csv"

# ----------------------------------------------------------------
# 결과 저장
# ----------------------------------------------------------------
OUTPUT_DIR = (
    PROJECT_DIR
    / "outputs"
    / "rag_common_evaluation"
    / MODEL_NAME
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# LLM-only 팀과 반드시 동일하게 유지
# ----------------------------------------------------------------
MAX_NEW_TOKENS = 300
DO_SAMPLE = False

# ----------------------------------------------------------------
# RAG 설정 (1차 Top-5 검색 후 리랭커로 최종 남길 문서 수 = 3)
# ----------------------------------------------------------------
RETRIEVAL_TOP_K = 3

# ----------------------------------------------------------------
# 공통 Semantic 평가 모델
# ----------------------------------------------------------------
SEMANTIC_MODEL_ID = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# ----------------------------------------------------------------
# 기본 확인
# ----------------------------------------------------------------
assert CANONICAL_CSV.exists(), f"평가 CSV가 없습니다: {CANONICAL_CSV}"
assert MAX_NEW_TOKENS == 300, "팀 공통 조건: MAX_NEW_TOKENS는 300으로 유지하세요."
assert DO_SAMPLE is False, "팀 공통 조건: DO_SAMPLE은 False로 유지하세요."

if not ADAPTER_DIR.exists():
    print(f"[안내] 현재 경로에 LoRA 어댑터 폴더가 없습니다: {ADAPTER_DIR}")
    print("       어댑터 파일을 넣기 전까지는 Base 모델(EXAONE)로 동작합니다.")

print("MODEL_NAME       :", MODEL_NAME)
print("MODEL_ID         :", MODEL_ID)
print("ADAPTER_DIR      :", ADAPTER_DIR)
print("CANONICAL_CSV    :", CANONICAL_CSV)
print("OUTPUT_DIR       :", OUTPUT_DIR)
print("MAX_NEW_TOKENS   :", MAX_NEW_TOKENS)
print("DO_SAMPLE        :", DO_SAMPLE)
print("RETRIEVAL_TOP_K  :", RETRIEVAL_TOP_K)

In [ ]:
# ============================================================
# 3. 공통 평가 데이터 로드 및 검증
# ============================================================

eval_df = pd.read_csv(
    CANONICAL_CSV
)

required_cols = [
    "의도",
    "문의 내용",
    "응답",
    "reference_responses_json",
]

missing = [
    col
    for col in required_cols
    if col not in eval_df.columns
]

assert not missing, (
    f"필수 컬럼이 없습니다: {missing}"
)

assert len(eval_df) == 981, (
    f"평가 데이터가 981개가 아닙니다: {len(eval_df)}"
)

# reference_responses_json 파싱 가능 여부 확인
for i, value in enumerate(
    eval_df["reference_responses_json"].head(10)
):
    refs = json.loads(value)
    assert isinstance(refs, list), (
        f"{i}번째 reference_responses_json이 list가 아닙니다."
    )
    assert len(refs) >= 1, (
        f"{i}번째 reference가 비어 있습니다."
    )

print("✅ 공통 평가 데이터 확인 완료")
print("행 수:", len(eval_df))
print("컬럼:", eval_df.columns.tolist())

## 4. RAG 전용 System Prompt

LLM-only 팀과 **기본 상담 규칙은 동일하게 유지**하고,
RAG에서만 필요한 아래 지시만 추가합니다.

- 검색된 정책 문서를 우선 근거로 사용
- 정책 문서에 없는 정보는 임의 생성 금지

프롬프트가 완전히 같을 필요는 없습니다.
RAG는 정책 Context를 입력받는 시스템이므로 이 차이는 실험 목적에 맞는 차이입니다.

In [ ]:
# ============================================================
# 4. RAG System Prompt
# ============================================================

RAG_SYSTEM_PROMPT = '''
당신은 쇼핑몰 고객센터 상담 AI입니다.

고객 문의와 문의 유형(Intent), 그리고 검색된 정책 문서를 참고하여
고객에게 전달할 최종 답변을 작성하세요.

규칙:
- 검색된 정책 문서의 내용을 우선 근거로 사용하세요.
- 정책 문서에 없는 정책, 금액, 기간, 연락처, 사실은 임의로 만들지 마세요.
- 검색된 문서만으로 확실히 답할 수 없는 내용은 추측하지 마세요.
- 한국어 존댓말을 사용하세요.
- 답변은 간결하고 명확하게 작성하세요.
- 불필요한 분석 과정은 출력하지 마세요.
- 고객에게 전달할 최종 답변만 작성하세요.
'''.strip()


def make_rag_user_text(
    intent,
    question,
    context_text,
):
    return f'''
문의 유형: {intent}

고객 문의:
{question}

검색된 정책 문서:
{context_text}
'''.strip()


print("=== RAG SYSTEM PROMPT ===")
print(RAG_SYSTEM_PROMPT)

In [ ]:
# ============================================================
# 5. Base LLM + 학습된 LoRA Adapter 로드 (호환성 패치 포함)
# ============================================================
import inspect
import transformers.masking_utils as masking_utils

_orig_create_causal_mask = masking_utils.create_causal_mask
_ccm_params = inspect.signature(_orig_create_causal_mask).parameters

def _patched_create_causal_mask(*args, **kwargs):
    if "input_embeds" in kwargs and "inputs_embeds" in _ccm_params:
        kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
    if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in _ccm_params.values()):
        kwargs = {k: v for k, v in kwargs.items() if k in _ccm_params}
    return _orig_create_causal_mask(*args, **kwargs)

masking_utils.create_causal_mask = _patched_create_causal_mask

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0} if torch.cuda.is_available() else "auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

if ADAPTER_DIR.exists():
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print("[완료] Base LLM + LoRA Adapter 결합 로드 완료")
else:
    model = base_model
    print("[완료] Base LLM 단독 로드 완료 (LoRA 어댑터 미적용 상태)")

model.eval()
model.config.use_cache = True

if torch.cuda.is_available():
    print("GPU Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

## 6. Retriever 연결

여기에는 **RAG 팀이 이미 만들어 둔 Embedding + Vector DB + Retriever 코드**를 넣습니다.

예를 들어 팀 코드가 Chroma/FAISS 등을 사용한다면
그 코드를 아래 셀에 넣고 최종적으로 반드시:

```python
retriever = ...
```

객체가 만들어지도록 하세요.

이 Notebook에서는 특정 Vector DB나 Embedding 모델을 새로 강제하지 않습니다.
그 부분 자체가 RAG 팀의 시스템 구성입니다.

### 중요한 조건
Retriever가 검색할 때 다음 정보는 사용하면 안 됩니다.

- 정답 `응답`
- `reference_responses_json`
- 사람이 미리 지정한 정답 chunk

질문을 이용해 실제 검색된 정책 문서만 사용해야 합니다.

In [ ]:
# ============================================================
# 6. RAG 팀의 Retriever 연결 코드 (BGE-m3 Top-5 + BGE-Reranker Top-3)
# ============================================================
import subprocess
import chromadb
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

CHROMA_DIR = PROJECT_DIR / "chroma_db"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 코랩 등에서 chroma_db가 비어있을 경우 자동으로 ingest.py 실행하여 41개 청크 생성
need_ingest = True
if CHROMA_DIR.exists():
    _client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    for _col in _client.list_collections():
        _cname = _col.name if hasattr(_col, "name") else str(_col)
        if _client.get_collection(_cname).count() >= 40:
            need_ingest = False
            break

if need_ingest and (PROJECT_DIR / "src" / "ingest.py").exists():
    print("-> chroma_db 초기화를 위해 src/ingest.py를 실행합니다...")
    subprocess.run(["python", str(PROJECT_DIR / "src" / "ingest.py")], check=True)

# 1) 1차 Bi-Encoder 임베딩(BAAI/bge-m3) 및 Chroma DB 로드
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
active_collection = "cs_policy_kb"
for col in client.list_collections():
    col_name = col.name if hasattr(col, "name") else str(col)
    if client.get_collection(col_name).count() > 0:
        active_collection = col_name
        break

vectorstore = Chroma(
    collection_name=active_collection,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# 1차 후보군은 Top-5로 검색
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2) 2차 Cross-Encoder 리랭커(BAAI/bge-reranker-v2-m3) 로드
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=device)

def rerank_retriever(query: str) -> list[str]:
    '''1차 Top-5 검색 후 리랭커 점수로 정렬하여 상위 Top-3 약관 텍스트를 반환합니다.'''
    cand_docs = base_retriever.invoke(query)
    if not cand_docs:
        return []
    pairs = [[query, doc.page_content] for doc in cand_docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(cand_docs, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in scored_docs[:RETRIEVAL_TOP_K]]

    formatted_chunks = []
    for doc in top_docs:
        meta = doc.metadata or {}
        raw_src = meta.get("source_file") or meta.get("source") or "정책문서"
        src = Path(str(raw_src)).name
        header_parts = [src]
        for key in ["section", "subsection", "article"]:
            if meta.get(key):
                header_parts.append(str(meta[key]))
        if meta.get("page_start") and meta.get("page_end"):
            header_parts.append(f"p.{meta['page_start']}~{meta['page_end']}")
        elif meta.get("page") is not None:
            header_parts.append(f"p.{int(meta['page']) + 1}")
        header = " | ".join(header_parts)
        formatted_chunks.append(f"[{header}]
{doc.page_content.strip()}")
    return formatted_chunks

retriever = rerank_retriever
print(f"[완료] Retriever + Reranker 연결 완료 (컬렉션: {active_collection}, 청크 수: {vectorstore._collection.count()}개)")

In [ ]:
# ============================================================
# 7. Retriever 결과 형식 통일 Helper
# ============================================================

def build_retrieval_query(
    intent,
    question,
):
    '''
    기본값은 고객 질문만 검색에 사용합니다.

    만약 실제 서비스 RAG가 Intent까지 검색어에 사용하도록
    설계되어 있다면 아래를 다음처럼 바꿀 수 있습니다.

    return f"{intent}\n{question}"

    단, 팀 실험 문서에 해당 방식을 명시하세요.
    '''
    return str(question).strip()


def retrieve_documents(
    intent,
    question,
):
    assert retriever is not None, (
        "retriever가 아직 연결되지 않았습니다."
    )

    query = build_retrieval_query(
        intent,
        question,
    )

    # LangChain Retriever의 대표 방식
    if hasattr(retriever, "invoke"):
        docs = retriever.invoke(query)

    # 기존 get_relevant_documents 방식 대응
    elif hasattr(
        retriever,
        "get_relevant_documents"
    ):
        docs = retriever.get_relevant_documents(
            query
        )

    # 직접 함수 형태도 대응
    elif callable(retriever):
        docs = retriever(query)

    else:
        raise TypeError(
            "Retriever 호출 방식을 확인할 수 없습니다."
        )

    # Top-k 강제
    docs = list(docs)[:RETRIEVAL_TOP_K]

    contexts = []

    for doc in docs:
        if hasattr(doc, "page_content"):
            text = doc.page_content

        elif isinstance(doc, dict):
            text = (
                doc.get("page_content")
                or doc.get("content")
                or doc.get("text")
                or str(doc)
            )

        else:
            text = str(doc)

        text = str(text).strip()

        if text:
            contexts.append(text)

    return query, contexts


print("✅ Retriever helper 준비 완료")

In [ ]:
# ============================================================
# 8. RAG Prompt + LLM 생성 함수 (정규식 후처리 포함)
# ============================================================

def rag_answer(
    intent,
    question,
):
    # --------------------------------------------------------
    # 1) 실제 Retriever 검색 (Top-5 -> Reranker Top-3)
    # --------------------------------------------------------
    retrieval_query, contexts = retrieve_documents(
        intent,
        question,
    )

    # 검색 결과가 없는 경우에도 임의로 정답 문서를 넣지 않음
    if contexts:
        context_text = "\n\n".join(
            [
                f"[정책 문서 {i}]\n{ctx}"
                for i, ctx in enumerate(
                    contexts,
                    start=1,
                )
            ]
        )
    else:
        context_text = "검색된 관련 정책 문서가 없습니다."

    # --------------------------------------------------------
    # 2) RAG Prompt 구성
    # --------------------------------------------------------
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": make_rag_user_text(
                intent,
                question,
                context_text,
            ),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # --------------------------------------------------------
    # 3) Tokenize
    # --------------------------------------------------------
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    # --------------------------------------------------------
    # 4) 팀 공통 생성 조건
    # --------------------------------------------------------
    start = time.perf_counter()

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generation_seconds = time.perf_counter() - start

    generated_tokens = generated[
        0,
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    # [후처리] 내부 문서 번호 참조 태그 노출 제거
    answer = re.sub(r"\s*[\[\(](?:참고|문서|정책\s*문서|정책)\s*\d+[^\]\)]*[\]\)]", "", answer).strip()

    return {
        "answer": answer,
        "retrieval_query": retrieval_query,
        "retrieved_contexts": contexts,
        "generation_seconds": generation_seconds,
        "generated_tokens": int(generated_tokens.shape[0]),
    }

print("[완료] rag_answer() 준비 완료 (max_new_tokens=300, do_sample=False)")

In [ ]:
# ============================================================
# 9. 먼저 1개 질문으로 RAG 동작 확인
# ============================================================

sample = eval_df.iloc[0]

sample_intent = str(
    sample["의도"]
)

sample_question = str(
    sample["문의 내용"]
)

print("Intent:", sample_intent)
print("질문  :", sample_question)

result = rag_answer(
    sample_intent,
    sample_question,
)

assert isinstance(
    result,
    dict
), "rag_answer() 반환값은 dict여야 합니다."

assert "answer" in result
assert "retrieved_contexts" in result

print("\n=== 검색 Query ===")
print(result["retrieval_query"])

print("\n=== 검색된 정책 Chunk ===")
print(
    "Chunk 수:",
    len(result["retrieved_contexts"])
)

for i, ctx in enumerate(
    result["retrieved_contexts"],
    start=1,
):
    print(f"\n--- Chunk {i} ---")
    print(ctx[:1000])

print("\n=== 최종 RAG 답변 ===")
print(result["answer"])

print(
    "\n생성 시간:",
    round(
        result["generation_seconds"],
        3
    ),
    "초"
)

print(
    "생성 토큰:",
    result["generated_tokens"]
)

## 10. 981개 전체 생성

1개 테스트 결과에서 다음을 반드시 확인한 뒤 실행하세요.

- 실제 정책 문서가 검색되는지
- 검색된 Chunk가 질문과 관련 있는지
- System Prompt에 Context가 들어가는지
- 최종 답변이 정상적으로 생성되는지
- `max_new_tokens=300`, `do_sample=False`인지

아래 셀은 중간 결과를 주기적으로 저장합니다.
중간에 실행이 끊겨도 생성된 결과를 확인할 수 있습니다.

In [ ]:
# ============================================================
# 10. 981개 RAG 답변 생성
# ============================================================

rows = []

PARTIAL_PATH = (
    OUTPUT_DIR
    / "rag_predictions_partial.csv"
)

FINAL_PREDICTION_PATH = (
    OUTPUT_DIR
    / "rag_predictions.csv"
)

start_all = time.perf_counter()

for i, (_, row) in enumerate(
    eval_df.iterrows(),
    start=1,
):
    intent = str(
        row["의도"]
    )

    question = str(
        row["문의 내용"]
    )

    result = rag_answer(
        intent,
        question,
    )

    answer = str(
        result["answer"]
    ).strip()

    contexts = [
        str(x)
        for x in result.get(
            "retrieved_contexts",
            []
        )
    ]

    rows.append({
        "index": i - 1,
        "의도": intent,
        "문의 내용": question,
        "응답": row["응답"],
        "reference_responses_json":
            row["reference_responses_json"],
        "generated_response": answer,
        "retrieval_query":
            result["retrieval_query"],
        "retrieved_contexts_json":
            json.dumps(
                contexts,
                ensure_ascii=False,
            ),
        "retrieved_context_count":
            len(contexts),
        "generation_seconds":
            result["generation_seconds"],
        "generated_tokens":
            result["generated_tokens"],
    })

    # 25개마다 중간 저장
    if (
        i % 25 == 0
        or i == len(eval_df)
    ):
        pd.DataFrame(
            rows
        ).to_csv(
            PARTIAL_PATH,
            index=False,
            encoding="utf-8-sig",
        )

    if (
        i % 50 == 0
        or i == len(eval_df)
    ):
        print(
            f"{i:>4} / "
            f"{len(eval_df)} 완료"
        )


rag_df = pd.DataFrame(
    rows
)

rag_df.to_csv(
    FINAL_PREDICTION_PATH,
    index=False,
    encoding="utf-8-sig",
)

total_minutes = (
    time.perf_counter()
    - start_all
) / 60


print("\n✅ RAG 981개 생성 완료")
print(
    "총 시간:",
    round(
        total_minutes,
        2
    ),
    "분"
)

print(
    "빈 답변:",
    rag_df[
        "generated_response"
    ]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "평균 검색 Chunk 수:",
    round(
        rag_df[
            "retrieved_context_count"
        ].mean(),
        2
    )
)

print(
    "평균 생성 시간:",
    round(
        rag_df[
            "generation_seconds"
        ].mean(),
        3
    ),
    "초"
)

print(
    "저장:",
    FINAL_PREDICTION_PATH
)

In [ ]:
# ============================================================
# 11. 생성용 LLM GPU 메모리 해제
# Semantic 평가 모델 로드를 위해 먼저 정리
# ============================================================

del model
del base_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ 생성 모델 GPU 메모리 정리 완료")

In [ ]:
# ============================================================
# 12. ★ LLM-only와 동일한 공통 평가 함수
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False,
)

chrf_metric = CHRF()


def extract_key_facts(text):
    text = str(text)

    facts = set()

    # {{placeholder}}
    facts.update(
        re.findall(
            r"\{\{[^{}]+\}\}",
            text
        )
    )

    # [메뉴]
    facts.update(
        re.findall(
            r"\[[^\[\]]+\]",
            text
        )
    )

    # 숫자
    facts.update(
        re.findall(
            r"\d+(?:[.,]\d+)?",
            text
        )
    )

    # 전화번호
    facts.update(
        re.findall(
            r"\d{2,4}-\d{3,4}-\d{4}",
            text
        )
    )

    # URL
    facts.update(
        re.findall(
            r"https?://[^\s]+",
            text
        )
    )

    return facts


def key_fact_f1(
    prediction,
    reference,
):
    pred_facts = extract_key_facts(
        prediction
    )

    ref_facts = extract_key_facts(
        reference
    )

    # 둘 다 핵심정보가 없으면
    # Key Fact 평가 대상에서 제외
    if (
        not pred_facts
        and not ref_facts
    ):
        return np.nan

    if (
        not pred_facts
        or not ref_facts
    ):
        return 0.0

    common = (
        pred_facts
        & ref_facts
    )

    precision = (
        len(common)
        / len(pred_facts)
    )

    recall = (
        len(common)
        / len(ref_facts)
    )

    if (
        precision
        + recall
        == 0
    ):
        return 0.0

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )


def key_fact_f1_best(
    prediction,
    references,
):
    scores = [
        key_fact_f1(
            prediction,
            ref
        )
        for ref
        in references
    ]

    valid = [
        x
        for x
        in scores
        if not np.isnan(x)
    ]

    if not valid:
        return np.nan

    return max(valid)


def rouge_l_best(
    prediction,
    references,
):
    return max(
        rouge.score(
            ref,
            prediction
        )["rougeL"].fmeasure
        for ref
        in references
    )


def chrf_best(
    prediction,
    references,
):
    return max(
        chrf_metric.sentence_score(
            prediction,
            [ref],
        ).score / 100
        for ref
        in references
    )


def has_bad_repetition(text):
    text = str(text).strip()

    if not text:
        return True

    sentences = [
        x.strip()
        for x in re.split(
            r"[.!?]\s*",
            text
        )
        if x.strip()
    ]

    if len(sentences) < 2:
        return False

    counts = {}

    for sentence in sentences:
        counts[sentence] = (
            counts.get(
                sentence,
                0
            )
            + 1
        )

    return (
        max(counts.values())
        >= 3
    )


def is_korean_response(text):
    korean_chars = re.findall(
        r"[가-힣]",
        str(text)
    )

    return (
        len(korean_chars)
        >= 5
    )


def is_clean_answer(text):
    text = str(text).strip()

    if not text:
        return False

    bad_patterns = [
        "<think>",
        "</think>",
        "assistant:",
        "system:",
        "user:",
    ]

    lower_text = (
        text.lower()
    )

    return not any(
        pattern in lower_text
        for pattern in bad_patterns
    )


print("✅ 공통 평가 함수 준비 완료")

In [ ]:
# ============================================================
# 13. Multi-reference Semantic Similarity
# ============================================================

semantic_model = SentenceTransformer(
    SEMANTIC_MODEL_ID,
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    ),
)


def semantic_scores_multi(
    predictions,
    refs_list,
):
    pred_embeddings = (
        semantic_model.encode(
            [
                str(x)
                for x
                in predictions
            ],
            batch_size=64,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
    )

    flat_refs = []
    spans = []
    cursor = 0

    for refs in refs_list:
        refs = [
            str(x)
            for x in refs
        ]

        start = cursor

        flat_refs.extend(
            refs
        )

        cursor += len(
            refs
        )

        spans.append(
            (
                start,
                cursor
            )
        )

    ref_embeddings = (
        semantic_model.encode(
            flat_refs,
            batch_size=64,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
    )

    scores = []

    for i, (
        start,
        end
    ) in enumerate(
        spans
    ):
        similarities = (
            pred_embeddings[i]
            @ ref_embeddings[
                start:end
            ].T
        )

        scores.append(
            float(
                similarities
                .max()
                .item()
            )
        )

    return scores


refs_list = [
    json.loads(x)
    for x in rag_df[
        "reference_responses_json"
    ]
]


rag_df[
    "semantic_similarity"
] = semantic_scores_multi(
    rag_df[
        "generated_response"
    ].tolist(),
    refs_list,
)


print(
    "✅ Semantic Similarity:",
    f"{rag_df['semantic_similarity'].mean() * 100:.2f}%"
)

In [ ]:
# ============================================================
# 14. 나머지 공통 평가
# ============================================================

rag_df[
    "key_fact_f1"
] = [
    key_fact_f1_best(
        pred,
        refs,
    )
    for pred, refs
    in zip(
        rag_df[
            "generated_response"
        ],
        refs_list,
    )
]


rag_df[
    "rougeL"
] = [
    rouge_l_best(
        pred,
        refs,
    )
    for pred, refs
    in zip(
        rag_df[
            "generated_response"
        ],
        refs_list,
    )
]


rag_df[
    "chrf"
] = [
    chrf_best(
        pred,
        refs,
    )
    for pred, refs
    in zip(
        rag_df[
            "generated_response"
        ],
        refs_list,
    )
]


rag_df[
    "repetition_free"
] = [
    not has_bad_repetition(x)
    for x in rag_df[
        "generated_response"
    ]
]


rag_df[
    "korean_response"
] = [
    is_korean_response(x)
    for x in rag_df[
        "generated_response"
    ]
]


rag_df[
    "clean_answer"
] = [
    is_clean_answer(x)
    for x in rag_df[
        "generated_response"
    ]
]


key_valid = (
    rag_df[
        "key_fact_f1"
    ].dropna()
)

semantic = (
    rag_df[
        "semantic_similarity"
    ].mean()
)

key_f1 = (
    key_valid.mean()
)

rouge_l = (
    rag_df[
        "rougeL"
    ].mean()
)

chrf = (
    rag_df[
        "chrf"
    ].mean()
)

repetition = (
    rag_df[
        "repetition_free"
    ].mean()
)

korean = (
    rag_df[
        "korean_response"
    ].mean()
)

clean = (
    rag_df[
        "clean_answer"
    ].mean()
)


# LLM-only와 동일한 프로젝트 공통 점수
cs_quality_score = (
    semantic * 55
    + rouge_l * 20
    + chrf * 15
    + repetition * 5
    + korean * 5
)


print("\n=== RAG 공통 평가 결과 ===")
print(
    f"Semantic Similarity : "
    f"{semantic * 100:.2f}%"
)
print(
    f"Key Fact F1         : "
    f"{key_f1 * 100:.2f}% "
    f"({len(key_valid)}/{len(rag_df)} 평가 가능)"
)
print(
    f"ROUGE-L             : "
    f"{rouge_l * 100:.2f}%"
)
print(
    f"chrF                : "
    f"{chrf * 100:.2f}%"
)
print(
    f"Repetition Free     : "
    f"{repetition * 100:.2f}%"
)
print(
    f"Korean Response     : "
    f"{korean * 100:.2f}%"
)
print(
    f"Clean Answer        : "
    f"{clean * 100:.2f}%"
)
print(
    f"CS Quality Score    : "
    f"{cs_quality_score:.2f}/100"
)

In [ ]:
# ============================================================
# 15. 결과 저장
# ============================================================

DETAIL_PATH = (
    OUTPUT_DIR
    / "rag_common_evaluation_detail.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "rag_common_evaluation_summary.csv"
)

CONFIG_PATH = (
    OUTPUT_DIR
    / "rag_evaluation_config.json"
)


rag_df.to_csv(
    DETAIL_PATH,
    index=False,
    encoding="utf-8-sig",
)


summary_df = pd.DataFrame([
    {
        "model_name": MODEL_NAME,
        "mode": "LLM+RAG",
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "retrieval_top_k": RETRIEVAL_TOP_K,
        "semantic_similarity_percent":
            semantic * 100,
        "key_fact_f1_percent":
            key_f1 * 100,
        "key_fact_valid_count":
            len(key_valid),
        "rougeL_percent":
            rouge_l * 100,
        "chrf_percent":
            chrf * 100,
        "repetition_free_percent":
            repetition * 100,
        "korean_response_percent":
            korean * 100,
        "clean_answer_percent":
            clean * 100,
        "cs_quality_score_100":
            cs_quality_score,
        "avg_generation_seconds":
            rag_df[
                "generation_seconds"
            ].mean(),
        "median_generation_seconds":
            rag_df[
                "generation_seconds"
            ].median(),
        "p95_generation_seconds":
            rag_df[
                "generation_seconds"
            ].quantile(0.95),
        "avg_generated_tokens":
            rag_df[
                "generated_tokens"
            ].mean(),
        "avg_retrieved_context_count":
            rag_df[
                "retrieved_context_count"
            ].mean(),
        "empty_response_count":
            rag_df[
                "generated_response"
            ]
            .fillna("")
            .str.strip()
            .eq("")
            .sum(),
    }
])


summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)


config = {
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "adapter_dir": str(ADAPTER_DIR),
    "canonical_csv": str(CANONICAL_CSV),
    "evaluation_rows": len(eval_df),
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "retrieval_top_k": RETRIEVAL_TOP_K,
    "semantic_model": SEMANTIC_MODEL_ID,
    "rag_system_prompt": RAG_SYSTEM_PROMPT,
    "quality_score_formula": (
        "semantic*55 + rougeL*20 + "
        "chrf*15 + repetition*5 + korean*5"
    ),
}

CONFIG_PATH.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


display(
    summary_df.round(4)
)


print("\n✅ 저장 완료")
print("상세 결과 :", DETAIL_PATH)
print("요약 결과 :", SUMMARY_PATH)
print("실험 설정 :", CONFIG_PATH)

# 실행 순서 요약

RAG 팀은 아래 순서대로 진행하면 됩니다.

1. **2번 셀**
   - `MODEL_NAME`
   - `ADAPTER_DIR`
   - `PROJECT_DIR`
   를 자기 환경에 맞게 수정

2. **3번 셀**
   - `validation_canonical_981.csv`가 정확히 981개인지 확인

3. **4번 셀**
   - RAG System Prompt 확인
   - 기본 상담 규칙은 LLM-only와 동일
   - 정책 Context 지시만 RAG 전용으로 추가

4. **5번 셀**
   - Base LLM + 자기 학습 Adapter 로드

5. **6번 셀**
   - 기존 Embedding / Vector DB / Retriever 연결
   - 최종적으로 `retriever` 객체 생성

6. **7~8번 셀**
   - Retriever → Context → Prompt → LLM 흐름 준비

7. **9번 셀**
   - 반드시 1개 질문 테스트
   - 검색 chunk와 최종 답변을 눈으로 확인

8. **10번 셀**
   - 이상 없으면 981개 전체 생성

9. **11번 셀**
   - 생성 모델 GPU 정리

10. **12~14번 셀**
    - LLM-only와 동일한 공통 평가 실행

11. **15번 셀**
    - 상세 CSV / 요약 CSV / 설정 JSON 저장

## 최종 비교할 공통 지표

LLM-only와 LLM+RAG는 다음 지표를 나란히 비교합니다.

- Semantic Similarity
- Key Fact F1
- ROUGE-L
- chrF
- Repetition Free
- Korean Response
- Clean Answer
- CS Quality Score

## RAG에서만 별도로 보는 지표

아래는 RAG 전용이므로 공통 품질 점수에는 섞지 않습니다.

- Retrieval Recall@k
- Groundedness
- 검색 Chunk 품질

## 절대 사용하지 않을 것

Retriever나 Prompt에서 다음 값을 검색 힌트로 사용하지 마세요.

- 정답 응답
- `reference_responses_json`
- 미리 지정한 정답 정책 Chunk

그래야 실제 `LLM-only vs LLM+RAG` 비교가 됩니다.